In [ ]:
# =========================
# General EDA + Auto Encoding
# =========================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# -------------------------
# 0. Load dataset
# -------------------------
# Example:
# df = pd.read_csv("your_file.csv")

# -------------------------
# 1. Basic Info
# -------------------------
print("Shape of dataset:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

# -------------------------
# 2. Handle Missing Values
# -------------------------
# Fill missing numeric with median
for col in df.select_dtypes(include=["int64", "float64"]).columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical with mode
for col in df.select_dtypes(include=["object", "category"]).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values handled.")

# -------------------------
# 3. Numerical Variables (Histograms + Boxplots)
# -------------------------
num_vars = df.select_dtypes(include=["int64", "float64"]).columns

if len(num_vars) > 0:
    fig, axes = plt.subplots(len(num_vars), 2, figsize=(12, 5*len(num_vars)))

    if len(num_vars) == 1:
        axes = [axes]  # fix for single numeric col

    for i, var in enumerate(num_vars):
        sns.histplot(df[var].dropna(), kde=True, ax=axes[i][0], color="skyblue")
        axes[i][0].set_title(f"Histogram of {var}")

        sns.boxplot(x=df[var].dropna(), ax=axes[i][1], color="lightgreen")
        axes[i][1].set_title(f"Boxplot of {var}")

    plt.tight_layout()
    plt.show()

# -------------------------
# 4. Categorical Variables (Countplots)
# -------------------------
cat_vars = df.select_dtypes(include=["object", "category"]).columns

if len(cat_vars) > 0:
    fig, axes = plt.subplots(len(cat_vars), 1, figsize=(10, 5*len(cat_vars)))

    if len(cat_vars) == 1:
        axes = [axes]  # fix for single categorical col

    for i, var in enumerate(cat_vars):
        sns.countplot(x=df[var], ax=axes[i], palette="pastel")
        axes[i].set_title(f"Countplot of {var}")
        axes[i].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

# -------------------------
# 5. Convert Categorical → Numeric (Auto Choice)
# -------------------------
print("\nConverting categorical variables to numeric...")

for col in cat_vars:
    unique_vals = df[col].nunique()
    
    if unique_vals <= 10:  
        # Use One-Hot Encoding if few categories
        df = pd.get_dummies(df, columns=[col], drop_first=True)
        print(f"Applied One-Hot Encoding on '{col}' (Unique values: {unique_vals})")
    else:
        # Use Label Encoding if too many categories
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        print(f"Applied Label Encoding on '{col}' (Unique values: {unique_vals})")

# -------------------------
# 6. Correlation Heatmap (after encoding)
# -------------------------
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (Numeric + Encoded Features)")
plt.show()

# -------------------------
# Final Dataset Ready
# -------------------------
print("\nFinal dataset preview:")
print(df.head())
